In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
import sys
import analysis_utils as utils


In [3]:
"""
first_spike_permutation.py
Performs circular permutation tests to evaluate whether specific neurons
are more likely to fire first in SWRs than expected by chance, accounting
for firing rate.
"""



# ------------------------------------------------------------
#  Utility Function
# ------------------------------------------------------------

def time_in_range(t_start: float, t_end: float, t: float):
    """Check if time t lies between start and end (inclusive)."""
    try:
        return (t_start <= t <= t_end)
    except TypeError:
        print("Non-numeric entry provided, defaulting to False")
        return False


# ------------------------------------------------------------
#  Data Loading Function
# ------------------------------------------------------------

def load_spike_data(
    time_dir: str = "./spike_times.npy",
    cluster_dir: str = "./spike_clusters.npy",
    label_dir: str = "./cluster_KSLabel.tsv"
):
    """Load spike times, cluster IDs, and cluster labels into a single DataFrame."""
    try:
        spike_times = np.load(time_dir).flatten() / 30000  # convert to seconds
    except FileNotFoundError:
        print(f"File not found: {time_dir}")
        sys.exit(1)

    try:
        spike_clusters = np.load(cluster_dir).flatten()
    except FileNotFoundError:
        print(f"File not found: {cluster_dir}")
        sys.exit(1)

    try:
        cluster_labels = pd.read_csv(label_dir, sep='\t')
    except FileNotFoundError:
        print(f"File not found: {label_dir}")
        sys.exit(1)

    cluster_labels = cluster_labels.rename(columns={'cluster_id': 'Cluster ID'})
    spike_df = pd.DataFrame({"Time": spike_times, "Cluster ID": spike_clusters})
    spike_df = pd.merge(spike_df, cluster_labels, on='Cluster ID', how='left')
    return spike_df


# # ------------------------------------------------------------
# #  Match spikes to SWR windows Function
# # ------------------------------------------------------------

# def match_times(df, swr_dir=None, swr_df=None, only_keep_good=True, progress=True):
#     """
#     Assigns each SWR its list of spike times and cluster IDs.

#     You can pass either:
#       - swr_dir: path to SWR CSV
#       - swr_df: already-loaded SWR DataFrame (faster for permutations)
#     """
#     if swr_df is None and swr_dir is None:
#         raise ValueError("Provide either swr_dir or swr_df")

#     spike_df = df.copy()

#     # Load SWR data if path is given
#     if swr_df is None:
#         try:
#             swr_df = pd.read_csv(swr_dir)
#         except FileNotFoundError:
#             print(f"File not found: {swr_dir}")
#             sys.exit(1)
#     else:
#         swr_df = swr_df.copy()

#     if only_keep_good:
#         spike_df = spike_df[spike_df["KSLabel"] == "good"]

#     swr_df["Spike Times (s)"] = None
#     swr_df["Cluster IDs"] = None
#     swr_df = swr_df.astype({"Spike Times (s)": "object", "Cluster IDs": "object"})

#     spike_times = np.array(spike_df["Time"])
#     cluster_ids = np.array(spike_df["Cluster ID"])

#     for idx in tqdm(range(len(swr_df)), desc="Checking SWR Data", disable=not progress):
#         mask = (swr_df["Start"][idx] <= spike_times) & (spike_times <= swr_df["Stop"][idx])
#         swr_df.at[idx, "Spike Times (s)"] = list(spike_times[mask])
#         swr_df.at[idx, "Cluster IDs"] = list(cluster_ids[mask])

#     return swr_df


# # ------------------------------------------------------------
# #  Analysis Functions
# # ------------------------------------------------------------

# def count_spikes(swr_df: pd.DataFrame, mode="all"):
#     """
#     Count spikes per cluster across SWRs.

#     Parameters
#     ----------
#     swr_df : pd.DataFrame
#         Must contain columns:
#         - "Spike Times (s)" : list/array of spike times
#         - "Cluster IDs"     : list/array of cluster IDs
#     mode : str 
#         "all"   → count all spikes within each SWR
#         "first" → count only the first spike cluster per SWR 

#     Returns
#     -------
#     dict : {cluster_id: count}
#     """

#     if mode not in ("first", "all"):
#         raise ValueError("mode must be 'first' or 'all'")

#     counts = {}

#     for _, row in swr_df.iterrows():
#         times = row["Spike Times (s)"]
#         clusters = row["Cluster IDs"]

#         # Skip empty SWRs
#         if times is None or len(times) == 0:
#             continue

#         times = np.array(times, dtype=float)
#         clusters = np.array(clusters, dtype=float)

#         if mode == "first":
#             # find first spike
#             first_cluster = clusters[np.argmin(times)]
#             counts[first_cluster] = counts.get(first_cluster, 0) + 1

#         elif mode == "all":
#             # count every spike event
#             unique, freqs = np.unique(clusters, return_counts=True)
#             for cid, f in zip(unique, freqs):
#                 counts[cid] = counts.get(cid, 0) + f

#     return counts


# def compute_mean_firing_rates(spike_df: pd.DataFrame, total_duration: float = None):
#     """Compute mean firing rate (Hz) for each cluster."""
#     if total_duration is None:
#         total_duration = spike_df["Time"].max()
#     counts = spike_df["Cluster ID"].value_counts()
#     firing_rates = counts / total_duration  # spikes per second
#     return firing_rates.to_dict()


In [30]:

# def circular_permutation_test_with_firing_rate(
#     swr_df: pd.DataFrame,
#     spike_df: pd.DataFrame,
#     tail: str = "two",  # or one for two-tailed
#     total_duration: float = None,
#     n_permutations: int = 1000,
#     shift_range_seconds: float = 3.0, # Range from -3.0s to 3.0s
#     progress: bool = True,
#     save_path: str = None,
#     mode: str = "all"
# ):
#     """
#     Circularly permute SWR windows and test which neurons are first more often than chance.
#     Shift is a random -3 to 3 seconds from SWR start times
#     Includes normalization by firing rate and finite-sample p-value correction.
#     """
    
#     if total_duration is None:
#         total_duration = spike_df["Time"].max()

#     # --- Compute firing rates and true first-spiker counts ---
#     firing_rates = compute_mean_firing_rates(spike_df, total_duration)
#     true_counts = count_spikes(swr_df, mode)

#     all_permuted_counts = {cid: [] for cid in true_counts.keys()}

#     # --- Create bounds for the shift ---
#     lower_bound = -shift_range_seconds
#     upper_bound = shift_range_seconds

#     # Generate 1000 random floats uniformly distributed between low and high bounds
#     shifts = np.random.uniform(low=lower_bound, high=upper_bound, size=n_permutations)

#     # --- Run circular permutations ---
#     for shift in tqdm(shifts, desc="Running permutations", disable=not progress):
#         shifted_swr = swr_df.copy()
        
#         # Apply the shift and handle circularity using modulo
#         shifted_swr["Start"] = (shifted_swr["Start"] + shift) % total_duration
#         shifted_swr["Stop"] = (shifted_swr["Stop"] + shift) % total_duration
        
#         # NOTE ON GAPS: 
#         # This code assumes that the modulo operation correctly handles 
#         # the shift even with gaps present, provided that the 'match_times' 
#         # function is smart enough to only consider spikes within actual 
#         # recorded time windows. 
#         # If 'match_times' relies on data existing across the entire duration,
#         # you need to ensure shifts don't land in the middle gap.

#         shifted_swr = match_times(spike_df, swr_df=shifted_swr, only_keep_good=False, progress=False)
#         perm_counts = count_first_spikes(shifted_swr)

#         for cid in all_permuted_counts.keys():
#             all_permuted_counts[cid].append(perm_counts.get(cid, 0))


#     # --- Build results with corrected p-values ---
#     results = []
    
#     for cid, true_val in true_counts.items():
#         perm_vals = np.array(all_permuted_counts[cid])
#         n_perm = len(perm_vals)

#         mean_perm = perm_vals.mean()
#         std_perm = perm_vals.std(ddof=1) + 1e-6
#         rate = firing_rates.get(cid, np.nan)

#         # ----- p-value selection -----
#         if tail == "one":
#             # One-tailed (right-side only: true_val > random expectation)
#             p_value = (np.sum(perm_vals >= true_val) + 1) / (n_perm + 1)

#         elif tail == "two":
#             # Two-tailed: counts permutations as extreme or more extreme
#             p_value = (
#                 np.sum(np.abs(perm_vals - mean_perm) >= np.abs(true_val - mean_perm)) + 1
#             ) / (n_perm + 1)

#         else:
#             raise ValueError("Parameter 'tail' must be 'one' or 'two'.")

#         # ----- Normalizations and z-score -----
#         normalized_true = true_val / rate if rate > 0 else np.nan
#         normalized_mean = mean_perm / rate if rate > 0 else np.nan
#         z_score = (true_val - mean_perm) / std_perm if len(perm_vals) > 1 else np.nan

#         results.append({
#             "Cluster ID": cid,
#             "Firing Rate (Hz)": rate,
#             "True Count": true_val,
#             "Mean Permuted": mean_perm,
#             "Normalized True": normalized_true,
#             "Normalized Mean": normalized_mean,
#             "Z-Score": z_score,
#             f"p-value ({tail}-tailed)": p_value,
#         })

#         # optional: save the results
#         result_df = pd.DataFrame(results)

#         if save_path is not None:
#             os.makedirs(os.path.dirname(save_path), exist_ok=True)
#             result_df.to_csv(save_path, index=False)

#         print(f"\n Results saved to: {save_path}")

#     return result_df, true_counts, all_permuted_counts


In [6]:
# Run analysis directly in notebook

spike_df = load_spike_data(
    r"/Users/kellyschulte/Project_swe4s/data/full_data/spike_times.npy",
    r"/Users/kellyschulte/Project_swe4s/data/full_data/spike_clusters.npy",
    r"/Users/kellyschulte/Project_swe4s/data/full_data/cluster_KSLabel.tsv"
)
swr_df = utils.match_times(spike_df, swr_dir=r"/Users/kellyschulte/Project_swe4s/data/full_data/SWRs_7744_Partnerintro_ca2.csv")

result_df, true_counts, all_permuted_counts = utils.circular_permutation_test_with_firing_rate(
    swr_df,
    spike_df,
    tail = "two",
    n_permutations = 1000,
    shift_range_seconds = 3.0, 
    save_path=r"/Users/kellyschulte/Project_swe4s/first_spike_permutation.csv",
    mode = "all"
)

100%|███████████████████████████████████████| 1000/1000 [03:25<00:00,  4.86it/s]


In [10]:

def plot_permutation_histogram(
    cid, all_permuted_counts, true_counts, binsize = 20, two_tailed=True
):
    """
    Plot the permutation distribution and compute statistical values for a neuron.

    Parameters
    ----------
    cid : int or float
        Cluster ID of the neuron to analyze.

    all_permuted_counts : dict
        Dictionary mapping each cluster ID to a list of permutation-derived
        spike counts. Example:
        {12: [4, 5, 3, ...], 18: [10, 12, 11, ...]}.

    true_counts : dict
        Dictionary mapping each cluster ID to its observed spike count in the
        real (non-permuted) dataset.
    
    binsize : int
        Number of bins for the histogram

    two_tailed : bool, optional
        If True (default), compute a two-tailed p-value based on deviation
        from the mean of the permutation distribution.
        If False, compute a one-tailed p-value where probability is
        based on permuted values greater than or equal to the observed count.


    Notes
    -----
    - Permutation distribution:
      Computed by randomly shifting spike times or SWR boundaries to estimate
      the spike count expected by chance.

    - One-tailed p-value:
        p = (count(perm >= true) + 1) / (N_perm + 1)

    - Two-tailed p-value:
        p = (count(|perm - mean| >= |true - mean|) + 1) / (N_perm + 1)

    - Z-score:
        z = (true - mean_perm) / std_perm
      A small epsilon is added to std to avoid division by zero.

    The function displays:
        * A histogram of permuted spike counts
        * A vertical line indicating the observed count
        * Text summary of true value, permutation mean, p-value, and z-score

    Returns
    -------
    None
        This function produces a plot and prints summary statistics.
    """
    perm_vals = np.array(all_permuted_counts[cid])
    true_val = true_counts[cid]
    mean_perm = np.mean(perm_vals)

    if two_tailed:
        diff_true = abs(true_val - mean_perm)
        diff_perm = abs(perm_vals - mean_perm)
        p_value = (
            np.sum(diff_perm >= diff_true) + 1
        ) / (len(perm_vals) + 1)
    else:
        p_value = (
            np.sum(perm_vals >= true_val) + 1
        ) / (len(perm_vals) + 1)

    std_perm = perm_vals.std(ddof=1) + 1e-6
    z_score = (true_val - mean_perm) / std_perm

    plt.figure(figsize=(6, 4))
    plt.hist(
        perm_vals, bins=binsize, color="lightblue", edgecolor="black"
    )
    plt.axvline(
        true_val, color="red", linestyle="--", linewidth=2,
        label=f"True = {true_val}"
    )
    plt.title(
        f"Cluster {cid} | "
        f"{'Two' if two_tailed else 'One'}-tailed "
        f"p = {p_value:.3f}, z = {z_score:.2f}"
    )
    plt.xlabel("Spike count (permutation)")
    plt.ylabel("Frequency")
    plt.legend()
    plt.tight_layout()
    plt.show()

    print(
        f"Cluster {cid}: true = {true_val}, "
        f"mean_perm = {mean_perm:.2f}, "
        f"p = {p_value:.3f}, z = {z_score:.3f}"
    )


In [11]:
# One-tailed (default)
# plot_permutation_histogram(42, all_permuted_counts, true_counts)

# Two-tailed
plot_permutation_histogram(85, all_permuted_counts, true_counts, binsize=10, two_tailed=True)

Cluster 85: true = 792, mean_perm = 247.44, p = 0.001, z = 12.189


/var/folders/23/xv4_6bp114d91jywfwvhcz3m0000gn/T/ipykernel_86728/1746107240.py:92: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
